In [ ]:
import arviz as az
import seaborn as sns
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls
import itertools
from IPython.display import display, Markdown
import pandas as pd

from emu_renewal.inputs import OUTPUTS_PATH, get_google_mobility, get_apple_mobility
from emu_renewal.utils import get_countries_by_continent

In [ ]:
set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "44255911"
countries = ls(job_path)
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)

In [ ]:
def get_mob_weights(iso3, mob_type):
    get_mob = get_google_mobility if mob_type == "g_mob" else get_apple_mobility
    mob = get_mob(iso3)
    idata = az.from_netcdf(job_path / iso3 / f"{mob_type}/idata_filtered.nc")
    weights = idata.posterior["mob_weights"].to_dataframe().unstack("mob_weights_dim_0")
    weights.columns = mob.columns
    return weights

In [ ]:
weights = get_mob_weights("GBR", "g_mob")

In [ ]:
comb_cols = [", ".join(comb) for comb in itertools.combinations(weights.columns, 2)]

In [ ]:
country_corrs = pd.DataFrame(columns=comb_cols)
for iso3 in all_countries:
    try:
        weights = get_mob_weights(iso3, "g_mob")
        for col1, col2 in itertools.combinations(weights.columns, 2):
            country_corrs.loc[iso3, f"{col1}, {col2}"] = round(weights[col1].corr(weights[col2]), 4)
    except:
        print(f"no mobility available for {iso3}")

In [ ]:
country_corrs.to_excel("google_corrs.xlsx")